# GALR genes vs EMT score — pan-cancer correlation analysis

Correlates GALR1/GALR2/GALR3 expression with a Hallmark EMT gene-set
score across TCGA cancer types (Spearman correlation, per cancer type,
with BH-FDR correction applied globally across all tests).

**Inputs** (see `DATA_DIR` below):
- `tcga_RSEM_gene_tpm_annotated.tsv` — TCGA pan-cancer TPM expression matrix
- `HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION.v2026.1.Hs.gmt` — MSigDB EMT gene set
- `TCGA_sample_metadata.tsv` — sample -> cancer type mapping
- `selected_cancers.txt` — one cancer type per line, to restrict the analysis

**Outputs** (written to `OUTPUT_DIR`):
- `GALR_EMT_corr.tsv` / `_pval.tsv` / `_fdr.tsv` / `_n.tsv` — correlation stats matrices
- `EMT_scores_GALR.txt` — per-sample EMT score
- `GALR_EMT_heatmap.png` — correlation heatmap

In [ ]:
import os

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

## Configuration

Edit these for your own machine/dataset.

In [ ]:
DATA_DIR = "data"
OUTPUT_DIR = "output"

TARGET_GENES = ["GALR1", "GALR2", "GALR3"]
FDR_CUTOFF = 0.05
MIN_SAMPLES_PER_GROUP = 15

EXPRESSION_FILE = os.path.join(DATA_DIR, "tcga_RSEM_gene_tpm_annotated.tsv")
EMT_GENESET_FILE = os.path.join(DATA_DIR, "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION.v2026.1.Hs.gmt")
SAMPLE_METADATA_FILE = os.path.join(DATA_DIR, "TCGA_sample_metadata.tsv")
SELECTED_CANCERS_FILE = os.path.join(DATA_DIR, "selected_cancers.txt")

# TCGA study abbreviations for nicer axis labels on the heatmap
TCGA_ABBREVIATIONS = {
    "adrenocortical cancer": "ACC",
    "cervical & endocervical cancer": "CESC",
    "cholangiocarcinoma": "CHOL",
    "diffuse large B-cell lymphoma": "DBLC",
    "kidney papillary carcinoma": "KIRP",
    "brain lower grade glioma": "LGG",
    "mesothelioma": "MESO",
    "pheochromocytoma & paraganglioma": "PCPG",
    "sarcoma": "SARC",
    "uveal melanoma": "UVM",
    "breast invasive carcinoma": "BRCA",
    "lung adenocarcinoma": "LUAD",
    "lung squamous cell carcinoma": "LUSC",
    "colon adenocarcinoma": "COAD",
    "rectum adenocarcinoma": "READ",
    "prostate adenocarcinoma": "PRAD",
    "thyroid carcinoma": "THCA",
    "head & neck squamous cell carcinoma": "HNSC",
    "esophageal carcinoma": "ESCA",
    "stomach adenocarcinoma": "STAD",
    "liver hepatocellular carcinoma": "LIHC",
    "kidney clear cell carcinoma": "KIRC",
    "kidney papillary cell carcinoma": "KIRP",
    "bladder urothelial carcinoma": "BLCA",
    "uterine corpus endometrioid carcinoma": "UCEC",
    "ovarian serous cystadenocarcinoma": "OV",
    "skin cutaneous melanoma": "SKCM",
    "pancreatic adenocarcinoma": "PAAD",
    "glioblastoma multiforme": "GBM",
    "kidney chromophobe": "KICH",
    "thymoma": "THYM",
}

## Load expression data

In [ ]:
def load_expression(path: str) -> pd.DataFrame:
    """Load the TCGA TPM matrix, index by HGNC symbol, samples as columns."""
    expr_df = pd.read_csv(path, sep="\t", index_col=0)
    expr_df = expr_df.set_index("hgnc_symbol")
    expr_df = expr_df.drop(columns=["Ensembl_clean", "sample"], errors="ignore")
    expr_df = expr_df.select_dtypes(include=["number"])
    expr_df = expr_df[~expr_df.index.duplicated(keep="first")]
    expr_df.columns = expr_df.columns.str[:15]  # trim to 15-char TCGA sample barcode
    print("Expression matrix shape:", expr_df.shape)
    return expr_df

## Compute EMT score

In [ ]:
def compute_emt_score(expr_df: pd.DataFrame, gmt_path: str) -> pd.DataFrame:
    """Score each sample as the mean expression of the Hallmark EMT gene set."""
    emt_genes = []
    with open(gmt_path) as f:
        for line in f:
            if line.startswith("HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION"):
                emt_genes = line.strip().split("\t")[2:]

    emt_genes = [g for g in emt_genes if g in expr_df.index]
    print("EMT genes found in expression matrix:", len(emt_genes))

    emt_matrix = expr_df.loc[emt_genes]
    emt_score = emt_matrix.mean(axis=0)
    return emt_score.to_frame(name="EMT_score")

## Build merged table (target genes + EMT score + cancer type)

In [ ]:
def build_merged_table(
    expr_df: pd.DataFrame,
    target_genes: list,
    emt_score_df: pd.DataFrame,
    metadata_path: str,
) -> pd.DataFrame:
    """Join target-gene expression, EMT score and cancer type into one table (samples as rows)."""
    genes_present = [g for g in target_genes if g in expr_df.index]
    print("Target genes present in dataset:", genes_present)
    if not genes_present:
        raise ValueError(
            "None of the target genes were found in the expression matrix. "
            "Check gene symbol format (should be HGNC)."
        )

    sample_metadata = pd.read_csv(metadata_path, sep="\t", dtype=str)
    sample_metadata = sample_metadata.set_index("sample").rename(columns={"cancer": "CancerType"})

    merged = expr_df.loc[genes_present].T.copy()
    merged["EMT_score"] = emt_score_df["EMT_score"]
    merged["CancerType"] = merged.index.map(sample_metadata["CancerType"])

    missing = merged["CancerType"].isna().sum()
    print(f"Samples with missing cancer type: {missing}")
    merged = merged.dropna(subset=["CancerType"])
    print("Samples after filter:", merged.shape[0])

    # Z-score EMT score within each cancer type, so correlations aren't
    # driven by baseline EMT-score differences between cancer types.
    merged["EMT_z"] = merged.groupby("CancerType")["EMT_score"].transform(
        lambda x: (x - x.mean()) / x.std()
    )
    return merged, genes_present

## Correlate per cancer type

In [ ]:
def correlate_per_cancer(
    merged: pd.DataFrame,
    genes_present: list,
    selected_cancers: list,
    min_samples: int,
    fdr_cutoff: float,
):
    """Spearman-correlate each gene against EMT_z within each cancer type, with global BH-FDR."""
    corr_matrix = pd.DataFrame(index=genes_present, columns=selected_cancers, dtype=float)
    pval_matrix = pd.DataFrame(index=genes_present, columns=selected_cancers, dtype=float)
    n_matrix = pd.DataFrame(index=genes_present, columns=selected_cancers, dtype=float)

    for cancer in selected_cancers:
        subset = merged[merged["CancerType"] == cancer]
        for gene in genes_present:
            if gene in subset.columns and subset.shape[0] >= min_samples:
                valid = subset[[gene, "EMT_z"]].dropna()
                corr, pval = spearmanr(valid[gene], valid["EMT_z"])
                corr_matrix.loc[gene, cancer] = corr
                pval_matrix.loc[gene, cancer] = pval
                n_matrix.loc[gene, cancer] = len(valid)
            else:
                print(f"Skipped {gene} in {cancer}: n={subset.shape[0]}")

    # BH-FDR correction, applied globally across every gene x cancer test
    flat_pvals = pval_matrix.values.flatten()
    valid_mask = ~np.isnan(flat_pvals)
    fdr_flat = np.full(len(flat_pvals), np.nan)
    _, fdr_vals, _, _ = multipletests(flat_pvals[valid_mask], method="fdr_bh")
    fdr_flat[valid_mask] = fdr_vals

    fdr_matrix = pd.DataFrame(
        fdr_flat.reshape(pval_matrix.shape), index=pval_matrix.index, columns=pval_matrix.columns
    )
    significant_mask = fdr_matrix < fdr_cutoff

    return corr_matrix, pval_matrix, fdr_matrix, significant_mask

## Plot heatmap

In [ ]:
def plot_heatmap(corr_matrix: pd.DataFrame, significant_mask: pd.DataFrame, out_path: str):
    """Heatmap of Spearman r, with a marker over non-significant (FDR >= cutoff) cells."""
    labels = [TCGA_ABBREVIATIONS.get(c, c) for c in corr_matrix.columns]
    corr_matrix = corr_matrix.copy()
    significant_mask = significant_mask.copy()
    corr_matrix.columns = labels
    significant_mask.columns = labels

    plt.figure(figsize=(max(10, len(labels) * 0.8), 5))
    ax = sns.heatmap(
        corr_matrix.astype(float),
        cmap="PiYG",
        center=0,
        vmin=-1,
        vmax=1,
        annot=True,
        fmt=".2f",
        linewidths=0.5,
        cbar_kws={"label": "Spearman r (vs EMT_z)"},
    )

    for i in range(corr_matrix.shape[0]):
        for j in range(corr_matrix.shape[1]):
            if not significant_mask.iloc[i, j]:
                ax.text(j + 0.5, i + 0.5, "x", ha="center", va="center", color="black", fontsize=11)

    plt.title("Spearman Correlation: Target Genes vs EMT Score (Z-normalized)", fontsize=13)
    plt.xticks(rotation=60, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.figtext(0.5, -0.04, "x = FDR >= 0.05 (BH correction, global)", ha="center", fontsize=10)
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

## Run the analysis

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

expr_df = load_expression(EXPRESSION_FILE)
emt_score_df = compute_emt_score(expr_df, EMT_GENESET_FILE)
emt_score_df.to_csv(os.path.join(OUTPUT_DIR, "EMT_scores_GALR.txt"), sep="\t", index=True)

In [ ]:
merged, genes_present = build_merged_table(
    expr_df, TARGET_GENES, emt_score_df, SAMPLE_METADATA_FILE
)

with open(SELECTED_CANCERS_FILE) as f:
    selected_cancers = f.read().strip().split("\n")
print("Selected cancers:", selected_cancers)

In [ ]:
corr_matrix, pval_matrix, fdr_matrix, significant_mask = correlate_per_cancer(
    merged, genes_present, selected_cancers, MIN_SAMPLES_PER_GROUP, FDR_CUTOFF
)

corr_matrix.to_csv(os.path.join(OUTPUT_DIR, "GALR_EMT_corr.tsv"), sep="\t")
pval_matrix.to_csv(os.path.join(OUTPUT_DIR, "GALR_EMT_pval.tsv"), sep="\t")
fdr_matrix.to_csv(os.path.join(OUTPUT_DIR, "GALR_EMT_fdr.tsv"), sep="\t")

print("\nCorrelation matrix:")
print(corr_matrix)
print("\nSignificant (FDR < 0.05):")
print(significant_mask)

## Plot and save the heatmap

In [ ]:
plot_heatmap(corr_matrix, significant_mask, os.path.join(OUTPUT_DIR, "GALR_EMT_heatmap.png"))